In [9]:
# وارد کردن کتابخانه‌ها
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split as test
from sklearn.pipeline import Pipeline as pl
from sklearn.compose import ColumnTransformer as ct
from sklearn.impute import SimpleImputer as si
from sklearn.preprocessing import OneHotEncoder as ohe
from sklearn.preprocessing import StandardScaler as std
from sklearn.linear_model import LogisticRegression as lr
from sklearn.tree import DecisionTreeClassifier as dtc
from sklearn.ensemble import RandomForestClassifier as rc

# تعیین Random Seed برای قابل تکرار بودن نتایج
RANDOM_STATE = 42

# خواندن داده 
df = pd.read_csv("bank-full.csv", sep=";")

# مشخص کردن Target
TARGET = "y"

# جدا کردن Featureها
X = df.drop(columns=TARGET)

# جدا کردن Target
y = df[TARGET]

# حذف duration به دلیل در دسترس نبودن آن قبل از تماس
X = X.drop(columns=["duration"])

# نمایش ابعاد داده
print("X:", X.shape)
print("y:", y.shape)

X: (45211, 15)
y: (45211,)


In [2]:
# تقسیم داده به Train و بخش موقت
# 70 درصد Train و 30 درصد برای Validation + Test

X_train, X_temp, y_train, y_temp = test(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

# تقسیم بخش موقت به Validation و Test
# هر کدام تقریباً 15 درصد از کل Dataset خواهند بود

X_val, X_test, y_val, y_test = test(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

# نمایش اندازه هر بخش
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (31647, 15)
Validation: (6782, 15)
Test: (6782, 15)


In [3]:
# شناسایی Featureهای عددی
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# شناسایی Featureهای categorical
categorical_features = X_train.select_dtypes(
    include="object"
).columns.tolist()

print("Numeric Features:")
print(numeric_features)

print("\ncategorical Features:")
print(categorical_features)

Numeric Features:
['age', 'balance', 'day', 'campaign', 'pdays', 'previous']

categorical Features:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


C:\Users\hp\AppData\Local\Temp\ipykernel_2868\1660650269.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [4]:
# Pipeline مربوط به Featureهای عددی
numeric_pipeline = pl(steps=[
    
    # جایگزینی Missing Value با Median
    ("imputer", si(strategy="median")),
    
    # استانداردسازی Featureهای عددی
    ("scaler", std())
])


# Pipeline مربوط به Featureهای categorical
categorical_pipeline = pl(steps=[
    
    # جایگزینی Missing Value با پرتکرارترین مقدار
    ("imputer", si(strategy="most_frequent")),
    
    # تبدیل مقادیر categorical به Featureهای عددی
    # اگر Category جدیدی در Validation دیده شود، خطا ایجاد نمی‌شود
    ("encoder", ohe(
        handle_unknown="ignore",
        sparse_output=False
    ))
])


# ترکیب دو Pipeline
preprocessor = ct(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ]
)

In [5]:
# ساخت Logistic Regression
logistic_model = lr(
    max_iter=1000,
    random_state=RANDOM_STATE
)

# ساخت Pipeline کامل
# ابتدا preprocessing و سپس مدل اجرا می‌شود

logistic_pipeline = pl(steps=[
    ("preprocessing", preprocessor),
    ("model", logistic_model)
])

# آموزش مدل فقط روی Train
logistic_pipeline.fit(X_train, y_train)

# پیش‌بینی روی Validation
y_val_pred_logistic = logistic_pipeline.predict(X_val)

# دریافت احتمال کلاس yes
y_val_proba_logistic = logistic_pipeline.predict_proba(X_val)[:, 1]

In [7]:
# ساخت Decision Tree
decision_tree_model = dtc(
    random_state=RANDOM_STATE,
    max_depth=8
)

# ساخت Pipeline
decision_tree_pipeline = pl(steps=[
    ("preprocessing", preprocessor),
    ("model", decision_tree_model)
])

# آموزش مدل روی Train
decision_tree_pipeline.fit(X_train, y_train)

# پیش‌بینی روی Validation
y_val_pred_tree = decision_tree_pipeline.predict(X_val)

# دریافت احتمال کلاس yes
y_val_proba_tree = decision_tree_pipeline.predict_proba(X_val)[:, 1]

In [8]:
# ساخت Random Forest
random_forest_model = rc(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

# ساخت Pipeline
random_forest_pipeline = pl(steps=[
    ("preprocessing", preprocessor),
    ("model", random_forest_model)
])

# آموزش مدل روی Train
random_forest_pipeline.fit(X_train, y_train)

# پیش‌بینی روی Validation
y_val_pred_rf = random_forest_pipeline.predict(X_val)

# دریافت احتمال کلاس yes
y_val_proba_rf = random_forest_pipeline.predict_proba(X_val)[:, 1]